# Neural Networks — Exercises

10 graded exercises from universal approximation to scaling laws.

| Format | Description |
|---|---|
| **Problem** | Markdown cell with task description |
| **Your Solution** | Code cell with scaffolding |
| **Solution** | Complete reference solution with checks |

### Difficulty Levels

| Level | Exercises | Focus |
|---|---|---|
| ★ | 1–4 | Core mechanics — UAT, backprop, init, Adam |
| ★★ | 5–7 | Deeper theory — dropout, BatchNorm, residuals |
| ★★★ | 8–10 | AI applications — NTK, LoRA, scaling laws |

### Topic Map

| Topic | Exercise |
|---|---|
| Universal approximation width bound | 1 |
| Backpropagation by hand | 2 |
| Xavier and He initialisation derivation | 3 |
| Adam update and bias correction | 4 |
| Dropout ensemble averaging | 5 |
| BatchNorm backpropagation | 6 |
| Residual gradient lower bound | 7 |
| NTK kernel matrix | 8 |
| LoRA gradient analysis | 9 |
| Scaling laws and Chinchilla | 10 |


In [ ]:
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-6):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print('  expected:', np.asarray(expected))
        print('  got     :', np.asarray(got))
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def relu(z): return np.maximum(0, z)
def sigmoid(z): return 1 / (1 + np.exp(-z))

print('Setup complete.')


---

## Exercise 1 ★ — Universal Approximation Width Bound

Barron's theorem (1993) gives an explicit approximation rate for single-hidden-layer networks. For a function $f^* \in \mathcal{B}_C$ (Barron class with constant $C_f$), an $m$-neuron single-hidden-layer network $f_m$ satisfies:

$$\mathbb{E}[(f_m(\mathbf{x}) - f^*(\mathbf{x}))^2] \leq \frac{C_f^2}{m}$$

**Parts:**

(a) For $C_f = 10$ and target approximation error $\epsilon = 10^{-k}$ ($k=1,2,3,4$), compute the minimum width $m$ needed.

(b) Compare to the **classical non-parametric rate** for polynomials in dimension $d$: $O(\epsilon^{-d/2})$ terms. Compute for $d = 10, 100$ and $\epsilon = 0.01$.

(c) For Barron class, width $m$ is independent of dimension $d$ — explain why this constitutes an escape from the curse of dimensionality.

(d) Construct a function NOT in the Barron class: give a function with infinite spectral norm $C_f = \infty$ and verify it cannot be efficiently approximated.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 1: Your Solution
    
    C_f = 10.0
    
    # (a) Minimum width for each epsilon
    def min_width_barron(C_f, epsilon):
        # YOUR CODE HERE: m = C_f^2 / epsilon
        return None
    
    for k in [1, 2, 3, 4]:
        eps = 10**(-k)
        m = min_width_barron(C_f, eps)
        print(f'eps=1e-{k}: min width m = {m}')
    
    # (b) Classical polynomial rate
    def poly_terms(epsilon, d):
        # YOUR CODE HERE: O(eps^(-d/2))
        return None
    
    for d in [10, 100]:
        print(f'd={d}: polynomial terms needed = {poly_terms(0.01, d):.2e}')


In [ ]:
# Exercise 1: Solution

C_f = 10.0

def min_width_barron(C_f, epsilon):
    return int(np.ceil(C_f**2 / epsilon))

def poly_terms(epsilon, d):
    return epsilon**(-d/2)

header('Exercise 1: Universal Approximation Width Bound')

print('(a) Barron: minimum neurons for epsilon-approximation (C_f=10):')
for k in [1, 2, 3, 4]:
    eps = 10**(-k)
    m = min_width_barron(C_f, eps)
    print(f'  eps=1e-{k}: m = {m:,} neurons')

print('\n(b) Classical polynomial terms for eps=0.01:')
for d in [2, 5, 10, 50, 100]:
    p = poly_terms(0.01, d)
    print(f'  d={d:3d}: {p:.2e} polynomial terms')

check_true('Barron m=1000 for eps=0.01, C_f=10', min_width_barron(10.0, 0.01) == 10000)
check_true('Classical rate explodes: d=10 >> Barron', poly_terms(0.01, 10) > min_width_barron(10.0, 0.01))

# (d) Non-Barron function: Dirichlet kernel (infinite Fourier support)
print('\n(d) Non-Barron example: D_N(x) = sin((N+0.5)x)/sin(x/2)')
print('  This has infinite spectral norm as N->inf')
print('  Cannot be efficiently approximated by shallow networks')

print('\nTakeaway: Barron class functions need O(1/eps) neurons independent of dimension — ')
print('neural networks escape the curse of dimensionality for structured functions.')


---

## Exercise 2 ★ — Backpropagation by Hand

For a two-layer MLP $f(\mathbf{x}) = W^{[2]}\text{ReLU}(W^{[1]}\mathbf{x} + \mathbf{b}^{[1]}) + \mathbf{b}^{[2]}$:

$$W^{[1]} = \begin{pmatrix}1 & -1\\1 & 1\end{pmatrix}, \quad \mathbf{b}^{[1]} = \mathbf{0}, \quad W^{[2]} = \begin{pmatrix}1 & 1\end{pmatrix}, \quad \mathbf{b}^{[2]} = 0$$
$$\mathbf{x} = \begin{pmatrix}1\\0\end{pmatrix}, \quad y = 1$$

**Parts:**

(a) Perform the forward pass: compute $\mathbf{z}^{[1]}$, $\mathbf{h}^{[1]} = \text{ReLU}(\mathbf{z}^{[1]})$, $\hat{y} = W^{[2]}\mathbf{h}^{[1]} + b^{[2]}$.

(b) Compute MSE loss $\mathcal{L} = \frac{1}{2}(\hat{y} - y)^2$.

(c) Compute $\delta^{[2]} = \hat{y} - y$, $\partial\mathcal{L}/\partial W^{[2]}$, $\boldsymbol{\delta}^{[1]}$, $\partial\mathcal{L}/\partial W^{[1]}$.

(d) Verify with finite differences: check $\partial\mathcal{L}/\partial W^{[1]}_{11}$ numerically.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 2: Your Solution
    
    W1 = np.array([[1., -1.], [1., 1.]])
    b1 = np.zeros(2)
    W2 = np.array([[1., 1.]])
    b2 = np.zeros(1)
    x = np.array([1., 0.])
    y = 1.0
    
    # (a) Forward pass
    z1 = None   # YOUR CODE: W1 @ x + b1
    h1 = None   # YOUR CODE: ReLU(z1)
    yhat = None # YOUR CODE: W2 @ h1 + b2
    print(f'z1={z1}, h1={h1}, yhat={yhat}')
    
    # (b) Loss
    loss = None  # YOUR CODE
    print(f'loss = {loss}')
    
    # (c) Backward pass
    delta2 = None    # YOUR CODE: yhat - y (scalar)
    dW2    = None    # YOUR CODE: outer product
    dh1    = None    # YOUR CODE: W2.T * delta2
    delta1 = None    # YOUR CODE: dh1 * (z1 > 0)
    dW1    = None    # YOUR CODE: outer product
    print(f'dW1 = {dW1}')
    print(f'dW2 = {dW2}')


In [ ]:
# Exercise 2: Solution

W1 = np.array([[1., -1.], [1., 1.]])
b1 = np.zeros(2)
W2 = np.array([[1., 1.]])
b2 = np.zeros(1)
x = np.array([1., 0.])
y = 1.0

# (a) Forward
z1 = W1 @ x + b1         # [1, 1]
h1 = np.maximum(0, z1)   # [1, 1]
yhat = (W2 @ h1 + b2)[0] # 2.0

# (b) Loss
loss = 0.5 * (yhat - y)**2  # 0.5

# (c) Backward
delta2 = yhat - y            # 1.0 (scalar)
dW2    = np.outer([delta2], h1)  # [[1, 1]]
dh1    = W2.T * delta2           # [[1], [1]]
delta1 = dh1[:, 0] * (z1 > 0).astype(float)  # [1, 1]
dW1    = np.outer(delta1, x)     # [[1,0],[1,0]]
db1    = delta1                   # [1, 1]

# (d) Numerical check
def compute_loss(W1_, W2_, x_, y_):
    z1_ = W1_ @ x_
    h1_ = np.maximum(0, z1_)
    yhat_ = (W2_ @ h1_)[0]
    return 0.5 * (yhat_ - y_)**2

eps = 1e-5
W1_p = W1.copy(); W1_p[0,0] += eps
W1_m = W1.copy(); W1_m[0,0] -= eps
num_grad_W1_00 = (compute_loss(W1_p, W2, x, y) - compute_loss(W1_m, W2, x, y)) / (2*eps)

header('Exercise 2: Backpropagation by Hand')
print(f'Forward: z1={z1}, h1={h1}, yhat={yhat:.4f}, loss={loss:.4f}')
check_close('delta2 = yhat - y', delta2, 1.0)
check_close('dW2 correct', dW2, np.array([[1., 1.]]))
check_close('dW1[0,0] analytic vs numeric', dW1[0,0], num_grad_W1_00, tol=1e-4)
print(f'dW1 = {dW1}')
print(f'dW2 = {dW2}')
print('\nTakeaway: Backprop is just the chain rule in reverse topological order — ')
print('the cost is one forward pass worth of computation, regardless of parameter count.')


---

## Exercise 3 ★ — Xavier and He Initialisation

**Parts:**

(a) Derive Xavier init $\sigma_W^2 = 2/(d_{\text{in}} + d_{\text{out}})$ from the condition that forward and backward variance are both preserved for tanh (assume $\mathbb{E}[\sigma'^2] = 1$).

(b) Derive He init $\sigma_W^2 = 2/d_{\text{in}}$ for ReLU (use $\mathbb{E}[\text{ReLU}'^2] = 1/2$).

(c) Simulate signal propagation through a 25-layer network: compare He+ReLU vs Xavier+ReLU vs Xavier+tanh. Plot activation std per layer.

(d) For ReLU, compute the exact variance of $\text{ReLU}(z)$ when $z \sim \mathcal{N}(0, \sigma^2)$. Verify your formula with Monte Carlo simulation.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 3: Your Solution
    
    # (a) Xavier derivation: forward condition sigma_W^2 * d_in = 1
    #                         backward condition sigma_W^2 * d_out = 1
    #                         compromise: sigma_W^2 = 2 / (d_in + d_out)
    
    def xavier_std(d_in, d_out):
        # YOUR CODE HERE
        return None
    
    def he_std(d_in):
        # YOUR CODE HERE
        return None
    
    print('Xavier std for (64, 128):', xavier_std(64, 128))
    print('He std for (64,):        ', he_std(64))
    
    # (c) Variance propagation experiment
    def propagate(n_layers, width, init_fn, activation):
        np.random.seed(42)
        x = np.random.randn(200, width)
        stds = [x.std()]
        for _ in range(n_layers):
            W = np.random.randn(width, width) * init_fn(width)  # init_fn returns std
            x = x @ W.T
            if activation == 'relu': x = np.maximum(0, x)
            elif activation == 'tanh': x = np.tanh(x)
            stds.append(x.std())
        return np.array(stds)
    
    he_r   = propagate(25, 64, he_std, 'relu')
    xav_r  = propagate(25, 64, xavier_std, 'relu')
    xav_t  = propagate(25, 64, xavier_std, 'tanh')
    print('He+ReLU std at layer 25:   ', he_r[-1])
    print('Xavier+ReLU std at layer 25:', xav_r[-1])
    print('Xavier+tanh std at layer 25:', xav_t[-1])


In [ ]:
# Exercise 3: Solution

def xavier_std(d_in, d_out=None):
    if d_out is None: d_out = d_in
    return np.sqrt(2.0 / (d_in + d_out))

def he_std(d_in, d_out=None):
    return np.sqrt(2.0 / d_in)

def propagate(n_layers, width, init_fn, activation):
    np.random.seed(42)
    x = np.random.randn(500, width)
    stds = [x.std()]
    for _ in range(n_layers):
        W = np.random.randn(width, width) * init_fn(width)
        x = x @ W.T
        if activation == 'relu': x = np.maximum(0, x)
        elif activation == 'tanh': x = np.tanh(x)
        stds.append(x.std())
    return np.array(stds)

he_r   = propagate(25, 64, he_std,     'relu')
xav_r  = propagate(25, 64, xavier_std, 'relu')
xav_t  = propagate(25, 64, xavier_std, 'tanh')

# (d) Exact variance of ReLU(z) for z ~ N(0, sigma^2)
# ReLU(z) = max(0,z): Var[max(0,z)] = sigma^2/2 (half of input variance)
sigma = 2.0
z_mc = np.random.randn(100000) * sigma
relu_var_mc = np.var(np.maximum(0, z_mc))
relu_var_theory = sigma**2 / 2

header('Exercise 3: Xavier and He Initialisation')
check_close('Xavier std(64,64)', xavier_std(64, 64), np.sqrt(2.0/128), tol=1e-8)
check_close('He std(64)', he_std(64), np.sqrt(2.0/64), tol=1e-8)
check_true('He+ReLU: std at layer 25 ≈ O(1)', 0.1 < he_r[-1] < 5)
check_true('Xavier+ReLU: std decays (vanishing)', xav_r[-1] < 0.1)
check_close('Var[ReLU(z)] = sigma^2/2 (MC vs theory)', relu_var_mc, relu_var_theory, tol=0.1)
print(f'Var[ReLU(z)] MC: {relu_var_mc:.4f}, theory: {relu_var_theory:.4f}')
print('\nTakeaway: Wrong init → exponential signal decay. He init compensates for ReLU killing half the neurons.')


---

## Exercise 4 ★ — Adam Update and Bias Correction

**Parts:**

(a) Implement one Adam update step. Given gradient $g = 0.1$ at step $t=1$ with $\beta_1=0.9$, $\beta_2=0.999$, $\epsilon=10^{-8}$, $\eta=10^{-3}$: compute $m_1$, $v_1$, $\hat{m}_1$, $\hat{v}_1$, and $\Delta\theta_1$.

(b) Show why bias correction matters: compare $m_1/\sqrt{v_1}$ (no correction) to $\hat{m}_1/\sqrt{\hat{v}_1}$ (corrected). By what factor do they differ?

(c) Run Adam for 3000 steps on $\mathcal{L}(\theta) = \theta^2$ starting from $\theta_0 = 5$. Plot convergence and find the step where $|\theta_t| < 0.01$.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 4: Your Solution
    
    # (a) One Adam step
    g = 0.1; t = 1; b1 = 0.9; b2 = 0.999; eps = 1e-8; lr = 1e-3
    m0, v0 = 0.0, 0.0  # initial moments
    
    m1 = None  # YOUR CODE: b1*m0 + (1-b1)*g
    v1 = None  # YOUR CODE: b2*v0 + (1-b2)*g**2
    mh = None  # YOUR CODE: bias-corrected m
    vh = None  # YOUR CODE: bias-corrected v
    delta_theta = None  # YOUR CODE: lr * mh / (sqrt(vh) + eps)
    
    print(f'm1={m1}, v1={v1}')
    print(f'mhat={mh}, vhat={vh}')
    print(f'delta_theta = {delta_theta}')
    
    # (b) Effect of bias correction
    no_correction = None  # YOUR CODE: lr * m1 / (np.sqrt(v1) + eps)
    print(f'Without correction: {no_correction}')
    print(f'With correction:    {delta_theta}')
    print(f'Ratio: {delta_theta / no_correction if no_correction else None}')


In [ ]:
# Exercise 4: Solution

g = 0.1; t = 1; b1 = 0.9; b2 = 0.999; eps = 1e-8; lr = 1e-3
m0, v0 = 0.0, 0.0

m1 = b1*m0 + (1-b1)*g
v1 = b2*v0 + (1-b2)*g**2
mh = m1 / (1 - b1**t)
vh = v1 / (1 - b2**t)
delta_theta = lr * mh / (np.sqrt(vh) + eps)
no_correction = lr * m1 / (np.sqrt(v1) + eps)

# (c) Adam on quadratic
theta = 5.0
m_s, v_s = 0.0, 0.0
thetas, losses = [theta], [theta**2]
first_small = None
for t in range(1, 3001):
    g = 2 * theta  # gradient of theta^2
    m_s = b1*m_s + (1-b1)*g
    v_s = b2*v_s + (1-b2)*g**2
    mh_ = m_s/(1-b1**t); vh_ = v_s/(1-b2**t)
    theta -= lr * mh_ / (np.sqrt(vh_) + eps)
    thetas.append(theta)
    losses.append(theta**2)
    if first_small is None and abs(theta) < 0.01:
        first_small = t

header('Exercise 4: Adam Update and Bias Correction')
check_close('m1 = beta1*0 + (1-beta1)*g', m1, 0.01)
check_close('v1 = beta2*0 + (1-beta2)*g^2', v1, 0.001 * (1-b2))
check_close('mhat = m1 / (1 - beta1^1)', mh, m1 / (1 - b1))
factor = delta_theta / no_correction
check_true('Bias correction increases step size at t=1', factor > 5)
print(f'\nBias correction factor at t=1: {factor:.2f}x')
print(f'Without correction: {no_correction:.6f}')
print(f'With correction:    {delta_theta:.6f}')
print(f'Adam reaches |theta|<0.01 at step {first_small}')
print('\nTakeaway: At t=1, m1 is only (1-beta1)=10% of the true gradient — bias correction is 10x!')


---

## Exercise 5 ★★ — Dropout as Ensemble Averaging

**Parts:**

(a) For a single-hidden-layer network with 4 units and dropout $p=0.5$: enumerate all $2^4=16$ sub-networks. Count how many keep exactly $k$ neurons active.

(b) Prove for a **one-hidden-layer linear** network that $\mathbb{E}_{\mathbf{m}}[f_{\mathbf{m}}(\mathbf{x})] = f_{(1-p)W}(\mathbf{x})$ (expected output = output of full network with scaled weights).

(c) Implement MC Dropout with 100 samples on a regression network. Compute predictive mean and standard deviation. Verify: std is higher for out-of-distribution inputs.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 5: Your Solution
    
    import itertools
    
    # (a) Enumerate sub-networks
    def count_subnetworks(n_units, p):
        """Count sub-networks with each number of active units."""
        # YOUR CODE HERE
        return None
    
    count_subnetworks(4, 0.5)
    
    # (c) MC Dropout
    np.random.seed(42)
    def mc_dropout_predict(x, W1, b1, W2, b2, p=0.3, T=100):
        """Returns mean and std of T forward passes with dropout."""
        # YOUR CODE HERE
        return None, None
    
    W1 = np.random.randn(20, 1) * 0.5
    b1 = np.zeros(20)
    W2 = np.random.randn(1, 20) * 0.5
    b2 = np.zeros(1)
    
    x_in  = np.array([[1.0]])  # in-distribution
    x_ood = np.array([[10.0]]) # OOD
    mean_in, std_in = mc_dropout_predict(x_in, W1, b1, W2, b2)
    mean_ood, std_ood = mc_dropout_predict(x_ood, W1, b1, W2, b2)
    print(f'In-dist  (x=1):  mean={mean_in}, std={std_in}')
    print(f'OOD      (x=10): mean={mean_ood}, std={std_ood}')


In [ ]:
# Exercise 5: Solution

import itertools

def count_subnetworks(n_units, p):
    counts = {k: 0 for k in range(n_units+1)}
    total = 0
    for mask in itertools.product([0,1], repeat=n_units):
        k = sum(mask)
        prob = ((1-p)**k) * (p**(n_units-k))
        counts[k] += 1
        total += 1
    return counts, total

def relu(z): return np.maximum(0, z)

def mc_dropout_predict(x, W1, b1, W2, b2, p=0.3, T=100):
    preds = []
    for _ in range(T):
        h = relu(x @ W1.T + b1)  # (1, d_h)
        mask = (np.random.rand(*h.shape) > p).astype(float)
        h = h * mask / (1 - p)
        preds.append((h @ W2.T + b2)[0, 0])
    return np.mean(preds), np.std(preds)

np.random.seed(42)
W1 = np.random.randn(20, 1) * 0.5
b1 = np.zeros(20)
W2 = np.random.randn(1, 20) * 0.5
b2 = np.zeros(1)

x_in  = np.array([[1.0]])
x_ood = np.array([[10.0]])
mean_in,  std_in  = mc_dropout_predict(x_in, W1, b1, W2, b2, T=500)
mean_ood, std_ood = mc_dropout_predict(x_ood, W1, b1, W2, b2, T=500)

counts, total = count_subnetworks(4, 0.5)

header('Exercise 5: Dropout as Ensemble')
print('Subnetwork counts by active units:')
for k, c in counts.items():
    print(f'  {k} active: {c}/{total} = {c/total:.3f}')
print(f'Total sub-networks: {total} (= 2^4)')
check_true('In-dist std < OOD std (MC Dropout)', std_in < std_ood)
print(f'In-dist  std: {std_in:.4f}')
print(f'OOD      std: {std_ood:.4f}')
print('\nTakeaway: Dropout trains 2^n sub-networks simultaneously; MC Dropout uses variance across them for uncertainty.')


---

## Exercise 6 ★★ — BatchNorm Backpropagation

**Parts:**

(a) Derive $\partial\mathcal{L}/\partial\mathbf{z}$ (gradient through BatchNorm) starting from $\partial\mathcal{L}/\partial\hat{\mathbf{z}}$.

(b) Verify that the BatchNorm gradient has zero mean and is orthogonal to $\hat{\mathbf{z}}$.

(c) Implement BatchNorm forward + backward from scratch and verify with finite differences.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 6: Your Solution
    
    class BatchNormManual:
        def __init__(self, d, eps=1e-5):
            self.gamma = np.ones(d)
            self.beta  = np.zeros(d)
            self.eps = eps
        def forward(self, z):
            # YOUR CODE HERE
            # Store: mu, var, z_hat
            return None
        def backward(self, dout):
            # YOUR CODE HERE
            # dz, dgamma, dbeta
            return None, None, None
    
    # Test
    np.random.seed(0)
    bn = BatchNormManual(d=4)
    z = np.random.randn(8, 4)
    y = bn.forward(z)
    dout = np.random.randn(8, 4)
    dz, dg, db = bn.backward(dout)
    print('dz mean (should be ~0):', dz.mean(0) if dz is not None else None)


In [ ]:
# Exercise 6: Solution

class BatchNormManual:
    def __init__(self, d, eps=1e-5):
        self.gamma = np.ones(d)
        self.beta  = np.zeros(d)
        self.eps = eps

    def forward(self, z):
        B = z.shape[0]
        self.z = z
        self.mu  = z.mean(0)
        self.var = z.var(0)
        self.z_hat = (z - self.mu) / np.sqrt(self.var + self.eps)
        self.B = B
        return self.gamma * self.z_hat + self.beta

    def backward(self, dout):
        B = self.B
        dgamma = (dout * self.z_hat).sum(0)
        dbeta  = dout.sum(0)
        dz_hat = dout * self.gamma
        # Full BN backward
        std_inv = 1.0 / np.sqrt(self.var + self.eps)
        dz = std_inv / B * (B * dz_hat
                            - dz_hat.sum(0)
                            - self.z_hat * (dz_hat * self.z_hat).sum(0))
        return dz, dgamma, dbeta

# Numerical gradient check
np.random.seed(0)
d = 4; B = 8
bn = BatchNormManual(d=d)
z0 = np.random.randn(B, d)
dout = np.random.randn(B, d)

y0 = bn.forward(z0)
dz, dg, db = bn.backward(dout)

# Numerical check for z[0,0]
eps_fd = 1e-5
z_p = z0.copy(); z_p[0,0] += eps_fd
bn.forward(z_p); Lp = (bn.forward(z_p) * dout).sum()
z_m = z0.copy(); z_m[0,0] -= eps_fd
bn.forward(z_m); Lm = (bn.forward(z_m) * dout).sum()
num_dz00 = (Lp - Lm) / (2 * eps_fd)

header('Exercise 6: BatchNorm Backpropagation')
check_true('dz has zero mean', np.allclose(dz.mean(0), 0, atol=1e-10))
check_true('dz orthogonal to z_hat (dot product ≈ 0)',
           np.allclose((dz * bn.z_hat).sum(0), 0, atol=1e-10))
print(f'dz[0,0] analytic: {dz[0,0]:.8f}')
print(f'dz[0,0] numeric:  {num_dz00:.8f}')
print('\nTakeaway: BN gradient removes mean-shift and variance-scale components — only orthogonal updates pass through.')


---

## Exercise 7 ★★ — Residual Connections: Gradient Lower Bound

For a residual network where each block satisfies $\|F^{[l]}(\mathbf{h})\| \leq \alpha\|\mathbf{h}\|$:

**Parts:**

(a) Prove: $\|\partial\mathcal{L}/\partial\mathbf{h}^{[0]}\| \geq (1-\alpha)^L\|\partial\mathcal{L}/\partial\mathbf{h}^{[L]}\|$.

(b) Compare to the non-residual bound: $\|\partial\mathcal{L}/\partial\mathbf{h}^{[0]}\| \leq \alpha^L\|\partial\mathcal{L}/\partial\mathbf{h}^{[L]}\|$.

(c) Simulate both networks: measure gradient norm at each layer for $L=50$, $\alpha=0.3$.

(d) Verify the Veit et al. (2016) ensemble interpretation: compute the expected path length distribution through a 10-layer ResNet and verify the mean is $L/2$.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 7: Your Solution
    
    def lower_bound_residual(alpha, L):
        # YOUR CODE: (1-alpha)^L
        return None
    
    def upper_bound_plain(alpha, L):
        # YOUR CODE: alpha^L
        return None
    
    alpha = 0.3; L = 50
    print(f'Residual lower bound (alpha={alpha}, L={L}): {lower_bound_residual(alpha, L)}')
    print(f'Plain upper bound (alpha={alpha}, L={L}): {upper_bound_plain(alpha, L)}')


In [ ]:
# Exercise 7: Solution

def lower_bound_residual(alpha, L):
    return (1 - alpha)**L

def upper_bound_plain(alpha, L):
    return alpha**L

alpha = 0.3; L = 50
d = 32
np.random.seed(42)

# Simulate gradient flow
Ws = [np.random.randn(d, d) * (alpha / np.sqrt(d)) for _ in range(L)]

# Plain network: product of Jacobians
g = np.ones(d) / np.sqrt(d)
plain_norms = [la.norm(g)]
for W in Ws:
    g = W.T @ g * 0.5  # ReLU: 50% pass
    plain_norms.append(la.norm(g))

# Residual network: sum includes identity
g_r = np.ones(d) / np.sqrt(d)
resid_norms = [la.norm(g_r)]
for W in Ws:
    g_r = g_r + W.T @ g_r * 0.1
    resid_norms.append(la.norm(g_r))

# Veit et al. ensemble: path length distribution for L=10
L_small = 10
total_paths = 2**L_small
path_lengths = np.array([sum(mask) for mask in
                         np.array([[int(b) for b in format(i, f'0{L_small}b')]
                                   for i in range(total_paths)])])
mean_path_length = path_lengths.mean()

header('Exercise 7: Residual Gradient Analysis')
print(f'Residual lower bound (alpha={alpha}, L={L}): {lower_bound_residual(alpha, L):.6f}')
print(f'Plain upper bound    (alpha={alpha}, L={L}): {upper_bound_plain(alpha, L):.2e}')
check_true('Residual gradient >> plain at layer 0',
           resid_norms[0] > plain_norms[0])
print(f'Plain gradient norm at layer 0:    {plain_norms[0]:.4f}')
print(f'Residual gradient norm at layer 0: {resid_norms[0]:.4f}')
check_close('Mean path length = L/2', mean_path_length, L_small/2, tol=0.01)
print(f'Mean ResNet path length: {mean_path_length:.1f} (= L/2 = {L_small/2})')
print('\nTakeaway: Residual connections convert gradient products to sums — the identity path provides a floor.')


---

## Exercise 8 ★★★ — Neural Tangent Kernel Matrix

**Parts:**

(a) Implement `ntk_matrix(X, W1, W2)` computing the $n\times n$ NTK for a 2-layer MLP.

(b) Verify the NTK is positive semi-definite (all eigenvalues $\geq 0$).

(c) Compute the NTK regression prediction and compare to actual gradient descent training. As network width increases, verify they converge.

(d) Show that the NTK becomes more stable (less change during training) with larger width.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 8: Your Solution
    
    np.random.seed(42)
    n, d_in, d_h = 10, 2, 8
    X = np.random.randn(n, d_in)
    y = np.sin(X[:, 0]) * np.cos(X[:, 1])
    
    W1 = np.random.randn(d_h, d_in) / np.sqrt(d_in)
    W2 = np.random.randn(1, d_h)   / np.sqrt(d_h)
    
    def ntk_matrix(X, W1, W2):
        """Compute NTK: K_ij = grad_theta f(xi)^T grad_theta f(xj)."""
        n = len(X)
        K = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                # YOUR CODE HERE
                # Compute grad for xi, grad for xj, take dot product
                K[i, j] = 0  # placeholder
        return K
    
    K = ntk_matrix(X, W1, W2)
    print('NTK shape:', K.shape)
    print('NTK symmetric:', np.allclose(K, K.T))
    print('Min eigenvalue:', la.eigvalsh(K).min())


In [ ]:
# Exercise 8: Solution

np.random.seed(42)
n, d_in, d_h = 12, 2, 32
X = np.random.randn(n, d_in)
y = np.sin(X[:, 0]) * np.cos(X[:, 1])
X_te = np.random.randn(20, d_in)

def relu(z): return np.maximum(0, z)

def get_grads(x, W1, W2):
    h = relu(W1 @ x)
    act = (W1 @ x > 0).astype(float)
    gW1 = np.outer(W2[0] * act, x).flatten()
    gW2 = h.flatten()
    return np.concatenate([gW1, gW2])

def ntk_matrix(X, W1, W2):
    n = len(X)
    K = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            gi = get_grads(X[i], W1, W2)
            gj = get_grads(X[j], W1, W2)
            K[i, j] = gi @ gj
    return K

W1 = np.random.randn(d_h, d_in) / np.sqrt(d_in)
W2 = np.random.randn(1, d_h)   / np.sqrt(d_h)
K0 = ntk_matrix(X, W1, W2)

# NTK regression
eps_reg = 1e-4
K_te = np.array([[get_grads(X_te[i], W1, W2) @ get_grads(X[j], W1, W2)
                  for j in range(n)] for i in range(20)])
alpha_ntk = la.solve(K0 + eps_reg * np.eye(n), y)
y_ntk = K_te @ alpha_ntk

# Train actual network with GD
W1_gd, W2_gd = W1.copy(), W2.copy()
for _ in range(2000):
    h = relu(W1_gd @ X.T)       # (d_h, n)
    yhat = (W2_gd @ h)[0]       # (n,)
    err = (yhat - y) / n
    dW2 = (err @ h.T)
    dh  = W2_gd.T * err[None, :] * (W1_gd @ X.T > 0)
    dW1 = dh @ X
    W1_gd -= 0.1 * dW1; W2_gd -= 0.1 * dW2
y_gd = (W2_gd @ relu(W1_gd @ X_te.T))[0]

eigs = la.eigvalsh(K0)
header('Exercise 8: NTK Matrix')
check_true('NTK is symmetric', np.allclose(K0, K0.T))
check_true('NTK is PSD (all eigenvalues >= 0)', np.all(eigs >= -1e-8))
print(f'NTK eigenvalues: min={eigs.min():.4f}, max={eigs.max():.2f}')
print(f'NTK pred (first 5): {y_ntk[:5].round(4)}')
print(f'GD  pred (first 5): {y_gd[:5].round(4)}')
print('\nTakeaway: NTK is always PSD by construction; infinite-width limit = exact kernel regression.')


---

## Exercise 9 ★★★ — LoRA Gradient Analysis

**Parts:**

(a) Derive $\partial\mathcal{L}/\partial B$ and $\partial\mathcal{L}/\partial A$ for a linear model $\hat{\mathbf{y}} = (W_0 + BA)\mathbf{X}$ with MSE loss.

(b) With $B_0 = 0$: show initial gradient $\partial\mathcal{L}/\partial B$ equals the gradient of the pretrained model (correct fine-tuning initialisation).

(c) Compare LoRA rank $r=1,4,8,16$ to full fine-tuning on a synthetic task. Find the minimum rank achieving near-optimal MSE.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 9: Your Solution
    
    np.random.seed(42)
    m, n, d_in = 32, 32, 20  # weight matrix m x n, data d_in features
    n_samples = 50
    
    W0 = np.random.randn(m, n) * 0.1
    X_data = np.random.randn(d_in, n_samples)
    
    # True update: rank-4 ground truth
    r_true = 4
    U_true = la.svd(np.random.randn(m, r_true), full_matrices=False)[0]
    V_true = la.svd(np.random.randn(n, r_true), full_matrices=False)[0]
    DeltaW_true = U_true @ np.diag([3,2,1,0.5]) @ V_true.T
    
    # Generate targets
    y_target = (W0 + DeltaW_true) @ X_data  # (m, n_samples)
    
    def lora_loss(B, A, W0, X, y):
        yhat = (W0 + B @ A) @ X
        return 0.5 * np.mean((yhat - y)**2)
    
    def lora_gradients(B, A, W0, X, y):
        """Returns (dB, dA) for LoRA MSE loss."""
        # YOUR CODE HERE
        return None, None
    
    # Test with r=4
    r = 4
    B = np.zeros((m, r))
    A = np.random.randn(r, n) * 0.01
    dB, dA = lora_gradients(B, A, W0, X_data, y_target)
    print('dB shape:', dB.shape if dB is not None else None)
    print('dA shape:', dA.shape if dA is not None else None)


In [ ]:
# Exercise 9: Solution

np.random.seed(42)
m, n, d_in = 32, 32, 20
n_samples = 50

W0 = np.random.randn(m, n) * 0.1
X_data = np.random.randn(d_in, n_samples)

# Note: we need W (m x n) to multiply X_data (n x n_samples), so W must be (m x d_in)
# Let's use d_in for n to avoid confusion: W: (m, d_in), X: (d_in, n_samples)
W0 = np.random.randn(m, d_in) * 0.1
r_true = 4
U_true, _, Vt_true = la.svd(np.random.randn(m, d_in), full_matrices=False)
U_true = U_true[:, :r_true]; Vt_true = Vt_true[:r_true, :]
DeltaW_true = U_true @ np.diag([3,2,1,0.5]) @ Vt_true
y_target = (W0 + DeltaW_true) @ X_data

def lora_train(r, n_steps=1000, lr=0.01):
    B = np.zeros((m, r))
    A = np.random.randn(r, d_in) * 0.01
    for _ in range(n_steps):
        yhat = (W0 + B @ A) @ X_data
        E = (yhat - y_target) / n_samples  # (m, n_samples)
        dB = E @ X_data.T @ A.T
        dA = B.T @ E @ X_data.T
        B -= lr * dB; A -= lr * dA
    return 0.5 * np.mean(((W0 + B @ A) @ X_data - y_target)**2)

# Full fine-tuning
W_ft = W0.copy()
for _ in range(1000):
    yhat_ft = W_ft @ X_data
    E_ft = (yhat_ft - y_target) / n_samples
    W_ft -= 0.01 * E_ft @ X_data.T
loss_ft = 0.5 * np.mean((W_ft @ X_data - y_target)**2)

header('Exercise 9: LoRA Gradient Analysis')
print(f'Full fine-tuning MSE: {loss_ft:.6f}')
print(f'Full params: {m*d_in:,}')
print()
print(f'{"Rank":>6} | {"Test MSE":>12} | {"Params":>10} | {"Savings":>8}')
print('-' * 45)
for r in [1, 2, 4, 8, 16]:
    loss = lora_train(r)
    params = r * (m + d_in)
    savings = m * d_in / params
    print(f'{r:>6} | {loss:>12.6f} | {params:>10,} | {savings:>7.1f}x')
print()
check_true('LoRA r=4 achieves near-optimal MSE (true rank=4)', lora_train(4) < 10*loss_ft)
print('\nTakeaway: LoRA rank r >= r_true achieves near-optimal MSE; r < r_true is suboptimal.')


---

## Exercise 10 ★★★ — Scaling Laws and Chinchilla Optimum

**Parts:**

(a) Fit power-law scaling: given synthetic loss vs. compute data with $\mathcal{L}(C) = A \cdot C^{-\gamma} + \varepsilon$, fit $A$, $\gamma$, $\varepsilon$ using log-log linear regression.

(b) Using the Chinchilla formula $\mathcal{L}(N, D) = A/N^\alpha + B/D^\beta + E$, find the compute-optimal $(N^*, D^*)$ for a budget $C = 10^{22}$ FLOPs numerically.

(c) Verify the Chinchilla rule: $D^*/N^* \approx 20$ tokens per parameter.

(d) Predict the final loss for 3 published models (GPT-3, Chinchilla 70B, LLaMA-7B) using the formula and rank them by predicted efficiency.

In [ ]:
# Scaffold preserved for interactive work.
if False:
    # Exercise 10: Your Solution
    
    # (a) Fit power-law scaling
    np.random.seed(42)
    C_data = np.logspace(18, 23, 30)
    true_A, true_gamma, true_eps = 10.0, 0.05, 1.69
    L_data = true_A * C_data**(-true_gamma) + true_eps + 0.01*np.random.randn(len(C_data))
    
    def fit_scaling_law(C, L):
        """Fit L = A * C^(-gamma) + eps via log-log regression."""
        # Hint: log(L - eps) = log(A) - gamma * log(C)
        # YOUR CODE HERE (you can fix eps=1.69 if you know it)
        return None, None  # A, gamma
    
    A_fit, gamma_fit = fit_scaling_law(C_data, L_data)
    print(f'Fitted A={A_fit}, gamma={gamma_fit}')
    print(f'True    A={true_A}, gamma={true_gamma}')


In [ ]:
# Exercise 10: Solution

np.random.seed(42)
C_data = np.logspace(18, 23, 30)
true_A, true_gamma, true_eps = 10.0, 0.05, 1.69
L_data = true_A * C_data**(-true_gamma) + true_eps + 0.01*np.random.randn(len(C_data))

def fit_scaling_law(C, L, eps=1.69):
    log_L_minus_eps = np.log(np.maximum(L - eps, 1e-6))
    log_C = np.log(C)
    # Linear regression: log(L-eps) = log(A) - gamma*log(C)
    X_reg = np.column_stack([np.ones_like(log_C), log_C])
    coeffs, _, _, _ = np.linalg.lstsq(X_reg, log_L_minus_eps, rcond=None)
    log_A, neg_gamma = coeffs
    return np.exp(log_A), -neg_gamma

A_fit, gamma_fit = fit_scaling_law(C_data, L_data)

# (b) Chinchilla optimum
A_c, alpha = 406.4, 0.34
B_c, beta  = 410.7, 0.28
E_c = 1.69

def chinchilla_loss(N, D):
    return A_c/N**alpha + B_c/D**beta + E_c

C_budget = 1e22
Ns = np.logspace(8, 12, 2000)
Ds = C_budget / (6 * Ns)
valid = Ds > 1e6
losses = chinchilla_loss(Ns[valid], Ds[valid])
opt = np.argmin(losses)
N_star = Ns[valid][opt]; D_star = Ds[valid][opt]

# (d) Compare models
models = [('GPT-3', 175e9, 300e9), ('Chinchilla 70B', 70e9, 1.4e12), ('LLaMA-7B', 7e9, 1e12)]

header('Exercise 10: Scaling Laws and Chinchilla')
check_close('Fitted gamma ≈ true gamma', gamma_fit, true_gamma, tol=0.02)
print(f'Fitted: A={A_fit:.2f}, gamma={gamma_fit:.4f}')
print(f'Optimal: N*={N_star:.2e}, D*={D_star:.2e}, D*/N*={D_star/N_star:.1f}')
check_true('D*/N* ≈ 15-25 tokens per param', 10 < D_star/N_star < 30)
print('\nModel comparison:')
for name, N, D in models:
    L = chinchilla_loss(N, D); C = 6*N*D
    print(f'  {name:<20}: L={L:.4f}, compute={C:.2e}, D/N={D/N:.0f}')
print('\nTakeaway: Chinchilla 70B achieves lower loss than GPT-3 at lower compute — ')
print('optimal training requires ~20 tokens/param, most models are undertrained.')


---

## What to Review After Finishing

- [ ] **UAT** — Barron's $O(1/m)$ rate, why this escapes the curse of dimensionality
- [ ] **Backprop** — chain rule on DAG, delta equations, why cost = forward pass
- [ ] **He init** — derive from ReLU variance, verify empirically in 25-layer network
- [ ] **Adam bias correction** — why $m_0=0$ underestimates gradient by $(1-\beta_1)$
- [ ] **Dropout** — inverted dropout, ensemble interpretation, MC Dropout uncertainty
- [ ] **BatchNorm** — train/eval gap, zero-mean gradient property
- [ ] **Residuals** — lower bound proof, Veit et al. ensemble, mean path length = L/2
- [ ] **NTK** — PSD proof, lazy training regime, kernel regression equivalence
- [ ] **LoRA** — gradient derivation, why B=0 init is correct, rank needed = true rank
- [ ] **Scaling laws** — power-law fit, Chinchilla optimal = 20 tokens/param

## References

1. Cybenko (1989). Approximation by superpositions of a sigmoidal function.
2. Barron (1993). Universal approximation bounds for neural networks.
3. Glorot & Bengio (2010). Understanding the difficulty of training deep feedforward networks.
4. He et al. (2015). Delving deep into rectifiers (He initialisation).
5. Kingma & Ba (2015). Adam: A method for stochastic optimization.
6. Ioffe & Szegedy (2015). Batch normalization.
7. He et al. (2016). Deep residual learning for image recognition.
8. Jacot et al. (2018). Neural tangent kernel.
9. Hu et al. (2022). LoRA: Low-rank adaptation of large language models.
10. Hoffmann et al. (2022). Training compute-optimal large language models (Chinchilla).
